# **PUC-Rio | ENG 4560 — Projeto Integrado VI**
# **Teste Temporal — C3**

---

Quatro partes, nesta ordem:

| Parte | O que faz |
|---|---|
| **1** | Resolve o modelo com 1min, 5min, 10min, 30min, 1h, 2h e 6h e calcula as métricas das rotas |
| **2** | Gráficos da Parte 1 |
| **3** | Resolve o modelo com diferentes tolerâncias de gap e calcula as mesmas métricas |
| **4** | Gráficos da Parte 3 |

As métricas incluem custo, gap, tempo, status, clientes atendidos, rotas,
subtours, distância, tempos operacionais, violações de jornada e frota ativada.

O modelo matemático permanece o mesmo da Aula 4.

In [ ]:
# =====================================================
# (1) AMBIENTE
# =====================================================
# Instala as bibliotecas no kernel atual. O pacote PuLP fornece o executável do
# CBC quando ele não está instalado no sistema.
%pip install -q pyomo highspy gurobipy pulp

In [ ]:
# =====================================================
# CAMINHOS PADRONIZADOS DA INSTÂNCIA
# =====================================================
INSTANCIA = "C3"

import os
from pathlib import Path

ARQUIVOS = ("nodes.csv", "D.npy", "Cvar.npy", "q.npy", "s.npy",
            "Tmov_h.npy", "params.json")
current_dir = Path.cwd().resolve()

# Funciona quando o Jupyter inicia na raiz, na pasta da instância, em dados ou
# em notebooks. A busca é local e limitada aos ancestrais próximos.
candidate_roots = []
for anchor in (current_dir, *list(current_dir.parents)[:4]):
    candidate_roots.extend((
        anchor if anchor.name.upper() == INSTANCIA else anchor / INSTANCIA,
        anchor / "Projeto_Dis_Fis" / INSTANCIA,
    ))

instance_dir = next(
    (folder for folder in candidate_roots
     if all((folder / "dados" / name).is_file() for name in ARQUIVOS)),
    None,
)
if instance_dir is None:
    attempted = "\n".join(f"- {folder / 'dados'}" for folder in candidate_roots)
    raise FileNotFoundError(
        f"Dados da instância {INSTANCIA} não encontrados. Pastas verificadas:\n{attempted}"
    )

base_dir = instance_dir / "dados"
PASTA_NOTEBOOKS = instance_dir / "notebooks"
PASTA_CONFIG = instance_dir / "configuracao"
PASTA_RESULTADOS = instance_dir / "resultados"
PASTA_TABELAS = PASTA_RESULTADOS / "tabelas"
PASTA_GRAFICOS = PASTA_RESULTADOS / "graficos"
PASTA_LOGS = PASTA_RESULTADOS / "logs"
PASTA_RESUMOS = PASTA_RESULTADOS / "resumos"
PASTA_CHECKPOINTS = PASTA_RESULTADOS / "checkpoints"

for folder in (PASTA_TABELAS, PASTA_GRAFICOS, PASTA_LOGS,
               PASTA_RESUMOS, PASTA_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

cvar_file = base_dir / "Cvar.npy"
license_file = PASTA_CONFIG / "gurobi.lic"
if license_file.is_file():
    for key in ("GRB_WLSACCESSID", "GRB_WLSSECRET", "GRB_LICENSEID"):
        os.environ.pop(key, None)
    os.environ["GRB_LICENSE_FILE"] = str(license_file.resolve())
    print(f"Licença Gurobi: {license_file}")
else:
    print("Licença Gurobi local não encontrada; será usada a licença disponível no ambiente.")

print(f"Instância {INSTANCIA}: {instance_dir}")
print(f"Dados: {base_dir}")
print(f"Resultados: {PASTA_RESULTADOS}")

LICENCA_PLENA = False
try:
    import gurobipy as gp
    _teste = gp.Model()
    _teste.setParam("OutputFlag", 0)
    _vars = _teste.addVars(2500, lb=0, ub=1)
    _teste.setObjective(gp.quicksum(_vars.values()))
    _teste.optimize()
    _teste.dispose()
    LICENCA_PLENA = True
    print("Licença Gurobi compatível com o tamanho de C3/C4.")
except Exception as exc:
    print(f"Gurobi indisponível ou com licença restrita: {str(exc)[:90]}")

In [ ]:
# =====================================================
# (3) LEITURA E PARÂMETROS DA INSTÂNCIA
# =====================================================
import numpy as np, pandas as pd, json, math, time
try:
    from IPython.display import display as _ipy
except ImportError:
    _ipy = print

def mostrar(o):
    try:
        _ipy(o)
    except Exception:
        print(o)

nodes = pd.read_csv(base_dir / "nodes.csv")
D = np.load(base_dir / "D.npy")
C = np.load(base_dir / "Cvar.npy")
q = np.load(base_dir / "q.npy")
s = np.load(base_dir / "s.npy")
with (base_dir / "params.json").open(encoding="utf-8") as arquivo:
    params = json.load(arquivo)
n = len(nodes)

assert D.shape == (n, n) and C.shape == (n, n)
assert q.shape == (n,) and s.shape == (n,)

# frota heterogênea — valores lidos da própria instância
vehicle_types = ["FIO", "VUC"]
Q = {
    "FIO": float(params["VEHICLES"]["Fiorino"]["Q_kg"]),
    "VUC": float(params["VEHICLES"]["VUC"]["Q_kg"]),
}
f = {
    "FIO": float(params["VEHICLES"]["Fiorino"]["custo_fixo_diario"]),
    "VUC": float(params["VEHICLES"]["VUC"]["custo_fixo_diario"]),
}
H = float(params["H_horas"])
v_kmh = float(params["v_kmh"])
T = D / v_kmh
Q_BASE, f_BASE, TIPOS_BASE = dict(Q), dict(f), list(vehicle_types)

print(f"Instância {INSTANCIA}: {n-1} clientes | demanda total {q[1:].sum():.1f} kg")
print("Frota: " + " | ".join(
    f"{k} Q={Q[k]:.0f}kg f=R${f[k]:.0f}" for k in vehicle_types
))
print(f"Jornada H = {H}h | velocidade {v_kmh} km/h")

In [ ]:
# =====================================================
# (4) MODELO — restrições idênticas às da Aula 4
# =====================================================
from pyomo.environ import (ConcreteModel, RangeSet, Set, Var, Binary,
                           Constraint, Objective, minimize, value)
from pyomo.opt import SolverFactory

def build_model(use_mtz=True):
    m = ConcreteModel()
    m.N = RangeSet(0, n-1); m.C = RangeSet(1, n-1)
    m.K = Set(initialize=TIPOS_BASE)
    m.A = [(i,j,k) for i in range(n) for j in range(n) for k in TIPOS_BASE if i != j]
    m.x = Var(m.A, domain=Binary)
    m.u = Var(m.C, bounds=(1, n-1))
    m.y = Var(m.K, domain=Binary)

    m.activate_vehicle = Constraint(m.K, rule=lambda m,k:
        sum(m.x[0,j,k] for j in m.C) <= m.y[k])
    m.activate_return = Constraint(m.K, rule=lambda m,k:
        sum(m.x[i,0,k] for i in m.C) <= m.y[k])
    m.obj = Objective(rule=lambda m: (sum(C[i,j]*m.x[i,j,k] for (i,j,k) in m.A)
                                      + sum(f_BASE[k]*m.y[k] for k in m.K)),
                      sense=minimize)
    m.out = Constraint(m.C, rule=lambda m,i:
        sum(m.x[i,j,k] for j in m.N if j != i for k in m.K) == 1)
    m.inn = Constraint(m.C, rule=lambda m,j:
        sum(m.x[i,j,k] for i in m.N if i != j for k in m.K) == 1)
    m.depot_balance = Constraint(m.K, rule=lambda m,k:
        sum(m.x[0,j,k] for j in m.C) == sum(m.x[i,0,k] for i in m.C))
    m.single_departure = Constraint(m.K, rule=lambda m,k:
        sum(m.x[0,j,k] for j in m.C) <= 1)
    m.capacity = Constraint(rule=lambda m:
        sum(q[i] for i in range(1,n)) <= sum(Q_BASE[k]*m.y[k] for k in m.K))
    m.flow_by_type = Constraint(m.C, m.K, rule=lambda m,i,k:
        (sum(m.x[i,j,k] for j in m.N if j != i)
         - sum(m.x[j,i,k] for j in m.N if j != i)) == 0)
    if use_mtz:
        def _mtz(m, i, j):
            if i == j: return Constraint.Skip
            return m.u[i] - m.u[j] + (n-1)*sum(m.x[i,j,k] for k in m.K) <= n-2
        m.mtz = Constraint(m.C, m.C, rule=_mtz)
    return m

_m = build_model()
N_BIN = len(_m.A)
N_VAR = N_BIN + (n-1) + len(TIPOS_BASE)
N_CON = len(list(_m.component_data_objects(Constraint)))
print(f"Binárias: {N_BIN} | variáveis: {N_VAR} | restrições: {N_CON}")

In [ ]:
# =====================================================
# (5) RESOLUÇÃO E MÉTRICAS DAS ROTAS
# =====================================================
from collections import defaultdict

_TL  = {"gurobi": "TimeLimit", "cbc": "seconds", "highs": "time_limit"}
_GAP = {"gurobi": "MIPGap",    "cbc": "ratioGap", "highs": "mip_rel_gap"}

def _fam(nome):
    for k in _TL:
        if k in nome: return k
    return None

def solvers_disponiveis(c=("gurobi_direct", "appsi_highs", "cbc")):
    ok = []
    for sv in c:
        try:
            opt = SolverFactory("cbc", executable=__import__("pulp").PULP_CBC_CMD().path) if sv == "cbc" else SolverFactory(sv)
            if opt.available(exception_flag=False): ok.append(sv)
        except Exception: pass
    return ok

DISPONIVEIS = solvers_disponiveis()
if not LICENCA_PLENA and N_VAR > 2000:
    DISPONIVEIS = [sv for sv in DISPONIVEIS if "gurobi" not in sv]
    print("Modelo acima do limite da licença restrita: Gurobi removido.")
if not DISPONIVEIS:
    raise RuntimeError("Nenhum solver disponível. Execute primeiro a célula de ambiente.")
SOLVER = DISPONIVEIS[0]
print("Solvers disponíveis:", DISPONIVEIS, "| usando:", SOLVER)

# ---------- reconstrução das rotas ----------
def extrair_rotas(arcos):
    '''Separa ciclos que passam pelo depósito (rotas) dos que não passam (subtours).'''
    rotas, subtours = [], []
    porv = defaultdict(list)
    for (i,j,k) in arcos: porv[k].append((i,j))
    for k, arcs in porv.items():
        rest = list(arcs)
        suc = defaultdict(list)
        for (i,j) in arcs: suc[i].append(j)
        for j0 in list(suc.get(0, [])):
            if (0, j0) not in rest: continue
            rota, cur, nxt = [0], 0, j0
            while True:
                rest.remove((cur, nxt)); rota.append(nxt); cur = nxt
                if cur == 0: break
                nxt = next((b for (a,b) in rest if a == cur), None)
                if nxt is None: break
            rotas.append((k, rota))
        while rest:
            i0, j0 = rest[0]; ciclo, cur = [i0], i0
            while True:
                nxt = next((b for (a,b) in rest if a == cur), None)
                if nxt is None: break
                rest.remove((cur, nxt)); ciclo.append(nxt); cur = nxt
                if cur == i0: break
            subtours.append((k, ciclo))
    return rotas, subtours

def metricas_da_solucao(m):
    '''Todas as métricas de rota pedidas, a partir de um modelo já resolvido.'''
    arcos = [(i,j,k) for (i,j,k) in m.A if value(m.x[i,j,k]) > 0.5]
    rotas, subtours = extrair_rotas(arcos)

    det, atendidos, km_tot, viol = [], set(), 0.0, 0
    t_desloc = t_serv = 0.0
    for idx, (k, r) in enumerate(rotas, start=1):
        cli = [x for x in r if x != 0]
        atendidos.update(cli)
        km = sum(D[r[a], r[a+1]] for a in range(len(r)-1))
        tm = sum(T[r[a], r[a+1]] for a in range(len(r)-1))
        ts = sum(s[i] for i in cli)
        km_tot += km; t_desloc += tm; t_serv += ts
        viol += (tm + ts > H + 1e-9)
        det.append({"rota": idx, "tipo": k, "n_clientes": len(cli), "km": km,
                    "t_desloc_h": tm, "t_servico_h": ts, "t_total_h": tm+ts,
                    "jornada": "OK" if tm+ts <= H+1e-9 else "VIOLA",
                    "sequencia": "-".join(map(str, r))})
    orfaos = sum(len(c)-1 for _, c in subtours)
    jornadas = [d["t_total_h"] for d in det]
    return {
        "custo": float(value(m.obj)),
        "clientes_atendidos": len(atendidos),
        "clientes_total": n-1,
        "pct_atendidos": 100*len(atendidos)/(n-1),
        "n_rotas": len(rotas),
        "n_subtours": len(subtours),
        "clientes_orfaos": orfaos,
        "km_total": km_tot,
        "t_desloc_h": t_desloc,
        "t_servico_h": t_serv,
        "t_total_h": t_desloc + t_serv,
        "jornada_max_h": max(jornadas) if jornadas else 0.0,
        "rotas_violando_H": viol,
        "veiculos": "+".join(k for k in m.K if value(m.y[k]) > 0.5) or "-",
        "custo_fixo": sum(f_BASE[k]*value(m.y[k]) for k in m.K),
    }, pd.DataFrame(det)

def resolver(time_limit, mip_gap=None, solver=None):
    '''UM solve, sempre limitado. Devolve (dict de métricas, DataFrame das rotas).'''
    sv = solver or SOLVER
    fam = _fam(sv)
    opt = (SolverFactory("cbc", executable=__import__("pulp").PULP_CBC_CMD().path)
           if sv == "cbc" else SolverFactory(sv))
    opt.options[_TL[fam]] = float(time_limit)
    if mip_gap is not None: opt.options[_GAP[fam]] = float(mip_gap)

    m = build_model()
    t0 = time.time()
    res = opt.solve(m, load_solutions=False)
    elapsed = time.time() - t0

    base = {"solver": sv, "time_limit": time_limit, "gap_pedido": mip_gap,
            "tempo_s": elapsed,
            "status": str(res.solver.termination_condition),
            "otimo": str(res.solver.termination_condition) == "optimal"}
    try:    base["bound"] = float(res.problem.lower_bound)
    except Exception: base["bound"] = float("nan")

    try:
        m.solutions.load_from(res)
        met, det = metricas_da_solucao(m)
    except Exception:
        met = {"custo": float("nan"), "clientes_atendidos": 0,
               "clientes_total": n-1, "pct_atendidos": 0.0, "n_rotas": 0,
               "n_subtours": 0, "clientes_orfaos": 0, "km_total": float("nan"),
               "t_desloc_h": float("nan"), "t_servico_h": float("nan"),
               "t_total_h": float("nan"), "jornada_max_h": float("nan"),
               "rotas_violando_H": 0, "veiculos": "-", "custo_fixo": float("nan")}
        det = pd.DataFrame()

    base.update(met)
    b, c = base["bound"], base["custo"]
    base["gap_pct"] = (100*abs(c-b)/abs(c)
                       if all(map(math.isfinite, (b, c))) and abs(c) > 1e-9
                       else float("nan"))
    return base, det

def rot(seg):
    if seg < 60:    return f"{seg:g}s"
    if seg < 3600:  return f"{seg/60:g}min"
    if seg < 86400: return f"{seg/3600:g}h"
    return f"{seg/86400:g}d"

print("Pronto. resolver(time_limit, mip_gap) devolve métricas + detalhe por rota.")

---
# **PARTE 1 — Varredura de limite de tempo**

Um solve por limite: 1min, 5min, 10min, 30min, 1h, 2h e 6h.

Assim que a otimalidade for provada, as linhas restantes são preenchidas com o
mesmo resultado sem novo solve — um limite maior não muda uma solução já provada
ótima. Cada linha traz as métricas completas das rotas obtidas.

In [ ]:
# =====================================================
# PARTE 1 — LIMITE DE TEMPO
# =====================================================

# ---------------- CONFIGURAÇÃO ----------------
TIME_LIMITS = [60, 300, 600, 1800, 3600, 7200, 21600]
#             1min 5min 10min 30min   1h    2h     6h
PULAR_APOS_OTIMO = True     # não repete solves depois de provar otimalidade
SOLVER_P1 = SOLVER
# ----------------------------------------------

print(f"Instância {INSTANCIA} ({n-1} clientes) | solver: {SOLVER_P1}")
print(f"Limites: {[rot(t) for t in TIME_LIMITS]}")
print(f"Pior caso: {rot(sum(TIME_LIMITS))} (só se nunca provar otimalidade)\n")

linhas_p1, detalhes_p1 = [], {}
ja_otimo = None

for tl in TIME_LIMITS:
    if PULAR_APOS_OTIMO and ja_otimo is not None:
        r = dict(ja_otimo); r["time_limit"] = tl
        r["observacao"] = f"igual a {rot(ja_otimo['time_limit'])} (ótimo já provado)"
        r["tempo_s"] = ja_otimo["tempo_s"]
        linhas_p1.append(r)
        print(f"{rot(tl):>6} | reaproveitado (ótimo provado em {rot(ja_otimo['time_limit'])})")
        continue

    print(f"{rot(tl):>6} | resolvendo...", end=" ", flush=True)
    met, det = resolver(tl, solver=SOLVER_P1)
    met["observacao"] = ""
    linhas_p1.append(met); detalhes_p1[tl] = det
    print(f"{met['status']:<14} | {met['tempo_s']:8.1f}s | "
          f"custo R$ {met['custo']:9.2f} | gap {met['gap_pct']:6.2f}% | "
          f"{met['clientes_atendidos']}/{met['clientes_total']} clientes | "
          f"{met['km_total']:7.1f} km")
    if met["otimo"]:
        ja_otimo = met

df_p1 = pd.DataFrame(linhas_p1)
df_p1["rotulo"] = df_p1["time_limit"].map(rot)
df_p1.to_csv(PASTA_TABELAS / f"P1_time_limit_{INSTANCIA}.csv", index=False)

COLS = ["rotulo","status","tempo_s","custo","bound","gap_pct","clientes_atendidos",
        "n_rotas","n_subtours","km_total","t_total_h","jornada_max_h",
        "rotas_violando_H","veiculos"]
mostrar(df_p1[COLS].round(2))

In [ ]:
# =====================================================
# PARTE 1 — detalhe das rotas de cada limite
# =====================================================
for tl, det in detalhes_p1.items():
    if det.empty:
        print(f"\n### {rot(tl)}: nenhuma solução viável encontrada"); continue
    print(f"\n### {rot(tl)} — {len(det)} rota(s)")
    mostrar(det.round(2))

---
# **PARTE 2 — Gráficos do limite de tempo**

In [ ]:
# =====================================================
# PARTE 2 — GRÁFICOS
# =====================================================
import matplotlib.pyplot as plt

d = df_p1.copy()
x = np.arange(len(d)); lab = d["rotulo"].tolist()
AZUL, VERDE, LARANJA, ROXO, VERM = "#1f6feb", "#0b8a3e", "#bf8700", "#8250df", "#d1242f"

fig, ax = plt.subplots(2, 3, figsize=(16, 8.5))

# (a) custo e bound
a = ax[0,0]
a.plot(x, d["custo"], "o-", color=AZUL, lw=2, label="Custo (incumbente)")
a.plot(x, d["bound"], "s--", color=VERM, lw=2, label="Bound")
a.fill_between(x, d["bound"], d["custo"], color="#ffd8a8", alpha=.55, label="Gap")
for xi, c in zip(x, d["custo"]):
    if np.isfinite(c): a.annotate(f"{c:.0f}", (xi, c), fontsize=7,
                                  xytext=(0,7), textcoords="offset points", ha="center")
a.set_xticks(x); a.set_xticklabels(lab); a.set_ylabel("R$")
a.set_title("(a) Custo total × limite de tempo"); a.legend(fontsize=8); a.grid(alpha=.3)

# (b) gap
a = ax[0,1]
b = a.bar(lab, d["gap_pct"], color=ROXO)
a.bar_label(b, fmt="%.2f%%", fontsize=8)
a.set_ylabel("Gap (%)"); a.set_title("(b) Gap ao encerrar"); a.grid(alpha=.3, axis="y")

# (c) tempo realmente gasto vs limite concedido
a = ax[0,2]
a.plot(x, d["time_limit"], "s--", color="#999", lw=2, label="Limite concedido")
a.plot(x, d["tempo_s"], "o-", color=VERDE, lw=2, label="Tempo gasto")
a.set_yscale("log"); a.set_xticks(x); a.set_xticklabels(lab)
a.set_ylabel("segundos (log)"); a.set_title("(c) Tempo concedido × consumido")
a.legend(fontsize=8); a.grid(alpha=.3, which="both")

# (d) clientes atendidos
a = ax[1,0]
b = a.bar(lab, d["clientes_atendidos"], color=VERDE)
a.bar_label(b, fmt="%.0f", fontsize=8)
a.axhline(n-1, color=VERM, ls=":", lw=2, label=f"total = {n-1}")
a.set_ylabel("clientes"); a.set_title("(d) Clientes atendidos")
a.legend(fontsize=8); a.grid(alpha=.3, axis="y")

# (e) quilometragem e nº de rotas
a = ax[1,1]
b = a.bar(lab, d["km_total"], color=LARANJA)
a.bar_label(b, fmt="%.0f", fontsize=8)
a.set_ylabel("km"); a.set_title("(e) Quilometragem total")
a2 = a.twinx()
a2.plot(x, d["n_rotas"], "o-", color=AZUL, lw=2, label="nº de rotas")
a2.set_ylabel("rotas", color=AZUL); a2.legend(fontsize=8, loc="lower right")
a.grid(alpha=.3, axis="y")

# (f) jornada da rota mais longa
a = ax[1,2]
cores = [VERM if v > H else VERDE for v in d["jornada_max_h"]]
b = a.bar(lab, d["jornada_max_h"], color=cores)
a.bar_label(b, fmt="%.2f h", fontsize=8)
a.axhline(H, color=VERM, ls=":", lw=2, label=f"H = {H}h")
a.set_ylabel("horas"); a.set_title("(f) Jornada da rota mais longa")
a.legend(fontsize=8); a.grid(alpha=.3, axis="y")

for a in ax.flat: a.tick_params(axis="x", labelsize=8)
plt.suptitle(f"PARTE 2 — Limite de tempo | {INSTANCIA} ({n-1} clientes) | {SOLVER_P1}",
             fontweight="bold")
plt.tight_layout(); plt.savefig(PASTA_GRAFICOS / f"P2_time_limit_{INSTANCIA}.png", dpi=150); plt.show()

val = d[np.isfinite(d["custo"])]
if len(val):
    melhor = val["custo"].min()
    print(f"Melhor custo obtido: R$ {melhor:.2f}")
    prim = val[np.isclose(val["custo"], melhor)].iloc[0]
    print(f"Primeiro limite que já entrega esse custo: {prim['rotulo']} "
          f"(gastou {prim['tempo_s']:.1f}s)")
    if d["otimo"].any():
        po = d[d["otimo"]].iloc[0]
        print(f"Otimalidade PROVADA a partir de {po['rotulo']} "
              f"(tempo real: {po['tempo_s']:.1f}s)")
    else:
        print("Nenhum limite provou otimalidade — reporte sempre o gap junto do custo.")

---
# **PARTE 3 — Varredura de tolerância de gap**

Agora o limite de tempo fica fixo e o que varia é a tolerância: dizer ao solver
"pare quando estiver a X% do ótimo". As mesmas métricas de rota são recalculadas
para cada tolerância.

A pergunta que esta parte responde: **quanto de tempo se economiza aceitando uma
solução um pouco pior — e quão pior ela é de fato?**

In [ ]:
# =====================================================
# PARTE 3 — TOLERÂNCIA DE GAP
# =====================================================

# ---------------- CONFIGURAÇÃO ----------------
GAPS = [0.20, 0.10, 0.05, 0.02, 0.01, 0.005]   # fração: 0.05 = 5%
TL_P3 = 3600        # teto de segurança por rodada (s)
SOLVER_P3 = SOLVER
# ----------------------------------------------

print(f"Instância {INSTANCIA} | solver: {SOLVER_P3} | teto por rodada: {rot(TL_P3)}")
print(f"Tolerâncias: {[f'{100*g:g}%' for g in GAPS]}\n")

linhas_p3, detalhes_p3 = [], {}
for g in GAPS:
    print(f"{100*g:5.1f}% | resolvendo...", end=" ", flush=True)
    met, det = resolver(TL_P3, mip_gap=g, solver=SOLVER_P3)
    met["gap_pedido_pct"] = 100*g
    linhas_p3.append(met); detalhes_p3[g] = det
    print(f"{met['status']:<14} | {met['tempo_s']:8.1f}s | "
          f"custo R$ {met['custo']:9.2f} | gap real {met['gap_pct']:6.2f}% | "
          f"{met['clientes_atendidos']}/{met['clientes_total']} clientes | "
          f"{met['km_total']:7.1f} km")

df_p3 = pd.DataFrame(linhas_p3)
melhor_custo = df_p3["custo"].min()
df_p3["acima_do_melhor_pct"] = 100*(df_p3["custo"] - melhor_custo)/melhor_custo
tmax = df_p3["tempo_s"].max()
df_p3["economia_tempo_pct"] = 100*(1 - df_p3["tempo_s"]/tmax)
df_p3.to_csv(PASTA_TABELAS / f"P3_gap_{INSTANCIA}.csv", index=False)

mostrar(df_p3[["gap_pedido_pct","status","tempo_s","custo","gap_pct",
               "acima_do_melhor_pct","economia_tempo_pct","clientes_atendidos",
               "n_rotas","km_total","t_total_h","jornada_max_h","veiculos"]].round(2))

print("\nAtenção ao ler a coluna 'status': com tolerância de 20% o solver também")
print("reporta 'optimal'. Isso significa 'ótimo dentro da tolerância PEDIDA', não")
print("'ótimo provado'. Nunca reporte o status sem o gap ao lado.")

In [ ]:
# =====================================================
# PARTE 3 — detalhe das rotas de cada tolerância
# =====================================================
for g, det in detalhes_p3.items():
    if det.empty:
        print(f"\n### gap {100*g:g}%: nenhuma solução viável"); continue
    print(f"\n### gap {100*g:g}% — {len(det)} rota(s)")
    mostrar(det.round(2))

---
# **PARTE 4 — Gráficos da tolerância de gap**

In [ ]:
# =====================================================
# PARTE 4 — GRÁFICOS
# =====================================================
d = df_p3.copy().sort_values("gap_pedido_pct", ascending=False)
lab = [f"{g:g}%" for g in d["gap_pedido_pct"]]
x = np.arange(len(d))

fig, ax = plt.subplots(2, 3, figsize=(16, 8.5))

# (a) tempo gasto por tolerância
a = ax[0,0]
b = a.bar(lab, d["tempo_s"], color=AZUL)
a.bar_label(b, fmt="%.1fs", fontsize=8)
a.set_xlabel("Tolerância pedida"); a.set_ylabel("segundos")
a.set_title("(a) Tempo de solver"); a.grid(alpha=.3, axis="y")

# (b) custo obtido
a = ax[0,1]
b = a.bar(lab, d["custo"], color=LARANJA)
a.bar_label(b, fmt="%.0f", fontsize=8)
lo, hi = d["custo"].min(), d["custo"].max()
a.set_ylim(lo - max((hi-lo)*.5, lo*.01), hi + max((hi-lo)*.5, hi*.01))
a.set_xlabel("Tolerância pedida"); a.set_ylabel("R$")
a.set_title("(b) Custo obtido"); a.grid(alpha=.3, axis="y")

# (c) o preço da pressa
a = ax[0,2]
b = a.bar(lab, d["acima_do_melhor_pct"], color=VERM)
a.bar_label(b, fmt="%.2f%%", fontsize=8)
a.set_xlabel("Tolerância pedida"); a.set_ylabel("% acima do melhor custo")
a.set_title("(c) Quanto se paga a mais"); a.grid(alpha=.3, axis="y")

# (d) clientes atendidos
a = ax[1,0]
b = a.bar(lab, d["clientes_atendidos"], color=VERDE)
a.bar_label(b, fmt="%.0f", fontsize=8)
a.axhline(n-1, color=VERM, ls=":", lw=2, label=f"total = {n-1}")
a.set_xlabel("Tolerância pedida"); a.set_ylabel("clientes")
a.set_title("(d) Clientes atendidos"); a.legend(fontsize=8); a.grid(alpha=.3, axis="y")

# (e) quilometragem e jornada máxima
a = ax[1,1]
b = a.bar(lab, d["km_total"], color=LARANJA)
a.bar_label(b, fmt="%.0f", fontsize=8)
a.set_xlabel("Tolerância pedida"); a.set_ylabel("km")
a.set_title("(e) Quilometragem e jornada")
a2 = a.twinx()
a2.plot(x, d["jornada_max_h"], "o-", color=ROXO, lw=2, label="jornada máx (h)")
a2.axhline(H, color=VERM, ls=":", lw=1.5)
a2.set_ylabel("horas", color=ROXO); a2.legend(fontsize=8, loc="lower right")
a.grid(alpha=.3, axis="y")

# (f) fronteira tempo × qualidade
a = ax[1,2]
a.scatter(d["tempo_s"], d["acima_do_melhor_pct"], s=95, color=ROXO, zorder=3)
a.plot(d["tempo_s"], d["acima_do_melhor_pct"], "-", color=ROXO, alpha=.4, zorder=2)
for _, r in d.iterrows():
    a.annotate(f"{r['gap_pedido_pct']:g}%", (r["tempo_s"], r["acima_do_melhor_pct"]),
               fontsize=8, xytext=(7,6), textcoords="offset points")
a.set_xscale("log")
a.set_xlabel("Tempo (s, log)"); a.set_ylabel("% acima do melhor custo")
a.set_title("(f) Fronteira tempo × qualidade"); a.grid(alpha=.3, which="both")

for a in ax.flat: a.tick_params(axis="x", labelsize=8)
plt.suptitle(f"PARTE 4 — Tolerância de gap | {INSTANCIA} ({n-1} clientes) | {SOLVER_P3}",
             fontweight="bold")
plt.tight_layout(); plt.savefig(PASTA_GRAFICOS / f"P4_gap_{INSTANCIA}.png", dpi=150); plt.show()

barato = d[d["acima_do_melhor_pct"] <= 1.0]
if len(barato):
    e = barato.loc[barato["tempo_s"].idxmin()]
    print(f"Melhor custo-benefício com perda ≤ 1%: tolerância de "
          f"{e['gap_pedido_pct']:g}% -> {e['tempo_s']:.1f}s, "
          f"custo R$ {e['custo']:.2f} ({e['acima_do_melhor_pct']:.2f}% acima do melhor)")
    print(f"Economia de tempo em relação à rodada mais longa: "
          f"{e['economia_tempo_pct']:.0f}%")

In [ ]:
# =====================================================
# EXPORTAÇÃO
# =====================================================
resumo = {"instancia": INSTANCIA, "clientes": n-1, "solver": SOLVER,
          "variaveis": N_VAR, "restricoes": N_CON,
          "melhor_custo_P1": float(df_p1["custo"].min()),
          "melhor_custo_P3": float(df_p3["custo"].min()),
          "otimo_provado_P1": bool(df_p1["otimo"].any())}
json.dump(resumo, open(PASTA_RESUMOS / f"RESUMO_{INSTANCIA}.json", "w"), indent=2, default=str)

print(json.dumps(resumo, indent=2, ensure_ascii=False, default=str))
print("\nArquivos gerados:")
for a in sorted(os.listdir(PASTA_RESULTADOS)):
    if a.startswith(("P1_","P2_","P3_","P4_","RESUMO_")): print("  ", a)

---
## Execução desta versão

Este arquivo está fixado na instância **C3**. Execute as células em
ordem. As tabelas, figuras e o resumo são gravados automaticamente na pasta da
própria instância, mesmo quando o notebook é aberto a partir da raiz do projeto.